# Differentiable markets on reusable QuantLib meshes

This example prices two independent assets, each with a call and a put, through `MultiAssetBatch`. The plan reuses QuantLib spatial operators, averaged payoffs, exercise events and the original ADI steps. Financial inputs arrive **on each price call**: the first asset has differentiable flat rates, the second has differentiable dated discount factors. Both have differentiable spot prices.

The mesh, strikes, dates and exercise schedules are fixed numerical/financial configuration. The derivatives therefore describe changing the pricing equations and initial state on that mesh. The small grid below illustrates the API; production prices require a suitable domain and refinement checks.

Run from an installed QuantLibTorch environment, or from this repository after building its native extension.

In [1]:
from datetime import date
from pathlib import Path
import math
import sys
import torch

root = Path.cwd() if (Path.cwd() / 'python').exists() else Path.cwd().parent
if (root / 'python').exists():
    sys.path.insert(0, str(root / 'python'))
from quantlibtorch import MultiAssetBatch, state_grid

torch.set_num_threads(1)
dtype = torch.float64
axes = [[math.log(s) for s in [40., 60., 80., 100., 120., 150., 200.]],
        [0., .02, .05, .1, .2]]
dates = [date(2025, 1, 1), date(2025, 7, 1), date(2026, 1, 1)]
assets = [dict(strikes=[95., 105.], option_types=['call', 'put']),
          dict(strikes=[98., 108.], option_types=['call', 'put'])]
settings = dict(axes=axes, time_steps=8, exercise='bermudan',
                exercise_dates=dates[1:], checkpoint_steps=3)
plan = MultiAssetBatch(assets, **settings)
print('Compatible asset groups:', plan.groups)

Compatible asset groups: ((0, 1),)


## Model fields and market inputs

The fields below are Heston drift and noise loadings on a fixed mesh. This path is the existing general latent-field API; use `FdHestonEngine` when you want the original complete Heston engine and its automatic meshes. `initial_state` contains log spot and initial variance, so `spots.log()` keeps spot sensitivities connected.

Financial tensors are CPU float64. Original QuantLib `FlatForward` and log-linear `DiscountCurve` objects supply the forward rates for each actual rollback interval. For CUDA plans, these rates transfer to the device with their autograd graph. Curves are rebuilt every call; active market graphs never enter the passive plan cache. A dated curve overrides its flat rate, and an active shadowed rate raises an error.

In [2]:
variance = state_grid(axes)[:, 1]
root_v, zero = variance.sqrt(), torch.zeros_like(variance)
kappa, theta, sigma, rho = 1.5, .04, .3, -.6
drift = (kappa * (theta - variance))[None, :, None]
diffusion = torch.stack((torch.stack((root_v, zero), -1),
                        torch.stack((sigma*rho*root_v, sigma*math.sqrt(1-rho*rho)*root_v), -1)), -2)[None]
drifts = drift.expand(2, *drift.shape)
diffusions = diffusion.expand(2, *diffusion.shape)
rates = torch.nn.Parameter(torch.tensor([.03, .01], dtype=dtype))
discounts = torch.nn.Parameter(torch.tensor([.981, .953], dtype=dtype))
spots = torch.nn.Parameter(torch.tensor([100., 103.], dtype=dtype))

def markets():
    return [dict(rate=rates[0], dividend_yield=rates[1]),
            dict(discount_curve=[(dates[0], 1.), (dates[1], discounts[0]),
                                 (dates[2], discounts[1])], dividend_yield=.015)]

def initial_state():
    return torch.stack((spots.log(), torch.full_like(spots, .045)), -1)

prices = plan.price(drifts, diffusions, initial_state(), markets=markets())
gradients = torch.autograd.grad(prices.sum(), (rates, discounts, spots))
print('Prices [assets, contracts]:', prices.detach().tolist())
for label, value in zip(['rate/yield gradients', 'discount-factor gradients', 'spot gradients'], gradients):
    print(label, value.tolist())

Prices [assets, contracts]: [[11.812356252252345, 9.765637953357839], [13.010966612557894, 9.248103646548568]]
rate/yield gradients [5.611612617211633, -26.834619868104042]
discount-factor gradients [13.358121428638794, -19.721226190410203]
spot gradients [0.2279441123122303, 0.2757328047592416]


## Reuse the plan with a fresh optimization graph

Each iteration rebuilds only the active initial state and market data. This tiny fitting loop is an API demonstration, not a convergence or parameter-identification claim. The first curve discount stays fixed at one; all curve dates remain fixed. `mode='adjoint'` uses first-order equation VJPs with checkpoint replay. Use `mode='full'` when higher reverse derivatives are needed.

An empty override dictionary inherits the corresponding asset's original market configuration. Set `discount_curve=None` explicitly if replacing a configured dated curve with a flat rate.

In [3]:
target = prices.detach() + torch.tensor([[.05, -.03], [.04, -.02]], dtype=dtype)
optimizer = torch.optim.SGD([rates, discounts], lr=1e-6)
for iteration in range(3):
    optimizer.zero_grad()
    current = plan.price(drifts, diffusions, initial_state(),
                         markets=markets(), mode='adjoint')
    loss = (current - target).square().mean()
    loss.backward()
    optimizer.step()
    print(f'iteration={iteration}, squared pricing error={loss.item():.8f}')

iteration=0, squared pricing error=0.00135000


iteration=1, squared pricing error=0.00133403


iteration=2, squared pricing error=0.00131826


## Optional CUDA price and market-gradient parity

The same financial inputs remain CPU tensors. Model fields, rollback and readout use CUDA, while graph-connected QuantLib forward rates are prepared on CPU and transferred together. CUDA requires checked row-diagonally-dominant systems and raises when those conditions fail. This example establishes matching behavior, not a speed advantage for small portfolios. Fixed cash dividends and exercise schedules retain the existing batch compatibility restrictions.

In [4]:
if torch.cuda.is_available():
    gpu_plan = MultiAssetBatch(assets, device='cuda', **settings)
    cpu = plan.price(drifts, diffusions, initial_state(), markets=markets())
    cpu_grad = torch.autograd.grad(cpu.sum(), (rates, discounts, spots))
    gpu = gpu_plan.price(drifts.cuda(), diffusions.cuda(), initial_state().cuda(), markets=markets())
    gpu_grad = torch.autograd.grad(gpu.sum(), (rates, discounts, spots))
    torch.testing.assert_close(gpu.cpu(), cpu, rtol=3e-11, atol=3e-11)
    for actual, expected in zip(gpu_grad, cpu_grad):
        torch.testing.assert_close(actual, expected, rtol=3e-9, atol=3e-9)
    print('CPU/CUDA prices and market/spot gradients agree.')
else:
    print('CUDA unavailable; CPU example completed.')

CPU/CUDA prices and market/spot gradients agree.
